# Analyze Emoji / Symbol Codes in DOJ Epstein Files

This notebook:
1. Downloads PDFs from the DOJ Epstein Files Transparency Act disclosures
2. Runs OCR via **marker-pdf** (GPU-accelerated)
3. Scans extracted text for emoji and special symbol characters
4. Produces a report of all emoji/symbol occurrences with surrounding context

**Before running:**
1. Go to Runtime → Change runtime type → Select **A100 GPU** (or T4 if unavailable)
2. Data source: [DOJ Epstein Library](https://www.justice.gov/epstein/doj-disclosures)

In [ ]:
# Cell 1: Install dependencies (weasyprint is required by marker-pdf)
!pip install -q marker-pdf weasyprint requests tqdm emoji

In [ ]:
# Cell 2: Configuration
import os

DOJ_BASE_URL = "https://www.justice.gov"
DISCLOSURES_URL = f"{DOJ_BASE_URL}/epstein/doj-disclosures"

# Process ALL datasets (1-12)
DATASETS_TO_PROCESS = list(range(1, 13))

# No sampling — process everything
SAMPLE_SIZE = None

# Local directories
PDF_DIR = "/content/epstein_pdfs"
RESULTS_DIR = "/content/epstein_results"
os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Will process datasets: {DATASETS_TO_PROCESS}")
print(f"Sample size per dataset: ALL")

In [ ]:
# Cell 3: Scrape file listings from DOJ disclosure pages
import requests
import re
import time

def get_pdf_links(dataset_num, max_pages=100):
    """Scrape PDF download links from a DOJ dataset page."""
    all_links = []
    
    for page in range(max_pages):
        url = f"{DISCLOSURES_URL}/data-set-{dataset_num}-files"
        if page > 0:
            url += f"?page={page}"
        
        resp = requests.get(url, timeout=30)
        if resp.status_code != 200:
            print(f"  Page {page}: HTTP {resp.status_code}, stopping.")
            break
        
        # Find PDF links - pattern: /epstein/files/DataSet.../EFTA*.pdf
        pdf_links = re.findall(
            r'href="([^"]*(?:EFTA|DataSet)[^"]*\.pdf)"',
            resp.text,
            re.IGNORECASE
        )
        
        if not pdf_links:
            # Try broader pattern for any PDF links on the page
            pdf_links = re.findall(
                r'href="(/epstein/[^"]*\.pdf)"',
                resp.text,
                re.IGNORECASE
            )
        
        if not pdf_links:
            print(f"  Page {page}: No PDF links found, stopping.")
            break
        
        for link in pdf_links:
            full_url = link if link.startswith("http") else f"{DOJ_BASE_URL}{link}"
            if full_url not in all_links:
                all_links.append(full_url)
        
        print(f"  Page {page}: found {len(pdf_links)} links (total: {len(all_links)})")
        time.sleep(1)  # Be polite to the server
    
    return all_links

# Collect PDF URLs for each dataset
all_pdf_urls = {}
for ds_num in DATASETS_TO_PROCESS:
    print(f"\nScraping Dataset {ds_num}...")
    links = get_pdf_links(ds_num)
    all_pdf_urls[ds_num] = links
    print(f"  Dataset {ds_num}: {len(links)} PDFs found")

total = sum(len(v) for v in all_pdf_urls.values())
print(f"\nTotal PDFs discovered: {total}")

In [ ]:
# Cell 4: Download PDFs
from tqdm import tqdm
import random

downloaded_files = []
download_errors = []

for ds_num, urls in all_pdf_urls.items():
    # Apply sampling if configured
    if SAMPLE_SIZE and len(urls) > SAMPLE_SIZE:
        urls_to_download = random.sample(urls, SAMPLE_SIZE)
        print(f"Dataset {ds_num}: Sampling {SAMPLE_SIZE} of {len(urls)} PDFs")
    else:
        urls_to_download = urls
        print(f"Dataset {ds_num}: Downloading all {len(urls)} PDFs")
    
    ds_dir = os.path.join(PDF_DIR, f"dataset_{ds_num}")
    os.makedirs(ds_dir, exist_ok=True)
    
    for url in tqdm(urls_to_download, desc=f"Dataset {ds_num}"):
        filename = url.split("/")[-1]
        filepath = os.path.join(ds_dir, filename)
        
        if os.path.exists(filepath):
            downloaded_files.append(filepath)
            continue
        
        try:
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()
            with open(filepath, "wb") as f:
                f.write(resp.content)
            downloaded_files.append(filepath)
        except Exception as e:
            download_errors.append({"url": url, "error": str(e)})
        
        time.sleep(0.5)  # Rate limiting

print(f"\nDownloaded: {len(downloaded_files)}")
print(f"Errors: {len(download_errors)}")
if download_errors:
    for e in download_errors[:5]:
        print(f"  {e['url']}: {e['error']}")

In [ ]:
# Cell 5: Initialize marker OCR
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Cell 6: Run marker OCR on all downloaded PDFs + strip DOJ website boilerplate
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.config.parser import ConfigParser
import json
import gc
import re as _re

config_parser = ConfigParser({"output_format": "markdown"})
models = create_model_dict()
converter = PdfConverter(config=config_parser.generate_config_dict(), artifact_dict=models)

# --- Boilerplate stripping ---
# The DOJ site wraps each PDF in an HTML page with a header (search bar,
# navigation) and a footer (social media links, About/Careers/etc).
# Marker OCR picks all of this up.  We strip it so only the document body
# remains.

# Patterns that mark the END of the header / START of real content
_HEADER_END_PATTERNS = [
    _re.compile(r"(?:^|\n)#{1,3}\s+.{5,}", _re.MULTILINE),        # first real heading
    _re.compile(r"\| Search\s*\|.*?\n(?:\|.*\n)*", _re.DOTALL),    # search-bar table
]
# Patterns that mark the START of the footer
_FOOTER_START_PATTERNS = [
    _re.compile(r"\[Social Media\]", _re.IGNORECASE),
    _re.compile(r"subscriber/new\).*Social Media", _re.IGNORECASE | _re.DOTALL),
    _re.compile(r"\[About\]\(file:///about\)", _re.IGNORECASE),
]

def strip_doj_boilerplate(text):
    """Remove DOJ website header/footer that marker extracts around the PDF."""
    # --- strip footer ---
    footer_pos = len(text)
    for pat in _FOOTER_START_PATTERNS:
        m = pat.search(text)
        if m and m.start() < footer_pos:
            footer_pos = m.start()
    text = text[:footer_pos].rstrip()

    # --- strip header ---
    # Look for the search-bar markdown table that appears in every file
    table_match = _re.search(r"\| Search\s*\|.*?(?:\n\|.*)*\n", text)
    if table_match:
        text = text[table_match.end():].lstrip()
    else:
        # Fallback: skip everything before the first markdown heading
        heading = _re.search(r"(?:^|\n)(#{1,3}\s+.{5,})", text)
        if heading:
            text = text[heading.start():].lstrip()

    return text

# --- Run OCR ---
ocr_results_raw = {}   # before stripping
ocr_results = {}       # after stripping
ocr_errors = []

print(f"Running marker OCR on {len(downloaded_files)} PDFs...\n")

for filepath in tqdm(downloaded_files, desc="OCR"):
    filename = os.path.basename(filepath)
    try:
        rendered = converter(filepath)
        raw_text = rendered.markdown
        ocr_results_raw[filename] = raw_text
        ocr_results[filename] = strip_doj_boilerplate(raw_text)
    except Exception as e:
        ocr_errors.append({"file": filename, "error": str(e)})

    # Periodic GPU cleanup
    if len(ocr_results) % 20 == 0:
        torch.cuda.empty_cache()
        gc.collect()

print(f"\nOCR complete: {len(ocr_results)} succeeded, {len(ocr_errors)} errors")
if ocr_errors:
    for e in ocr_errors[:5]:
        print(f"  {e['file']}: {e['error']}")

# Show how much boilerplate was removed
if ocr_results:
    sample_file = next(iter(ocr_results))
    raw_len = len(ocr_results_raw.get(sample_file, ""))
    clean_len = len(ocr_results.get(sample_file, ""))
    print(f"\nBoilerplate removal example ({sample_file}):")
    print(f"  Raw: {raw_len} chars -> Clean: {clean_len} chars ({raw_len - clean_len} removed)")

# Save both raw and cleaned OCR results
with open(os.path.join(RESULTS_DIR, "ocr_texts.json"), "w") as f:
    json.dump(ocr_results, f)
with open(os.path.join(RESULTS_DIR, "ocr_texts_raw.json"), "w") as f:
    json.dump(ocr_results_raw, f)
print(f"Saved OCR results to {RESULTS_DIR}/ocr_texts.json")

In [ ]:
# Cell 7: Scan for emoji and special symbol characters (filtering website noise)
import unicodedata
import emoji
import json
import re

# Common UI / typography symbols to IGNORE — these appear in standard
# web rendering, markdown list bullets, etc. and are not meaningful.
IGNORE_CODEPOINTS = {
    0x2713,  # ✓ CHECK MARK
    0x2714,  # ✔ HEAVY CHECK MARK
    0x25E6,  # ◦ WHITE BULLET
    0x25CF,  # ● BLACK CIRCLE
    0x25CB,  # ○ WHITE CIRCLE
    0x2022,  # • BULLET
    0x25AA,  # ▪ BLACK SMALL SQUARE
    0x25AB,  # ▫ WHITE SMALL SQUARE
    0x25A0,  # ■ BLACK SQUARE
    0x25A1,  # □ WHITE SQUARE
    0x2023,  # ‣ TRIANGULAR BULLET
    0x00A9,  # © COPYRIGHT SIGN
    0x00AE,  # ® REGISTERED SIGN
    0x2122,  # ™ TRADE MARK SIGN
    0x00A7,  # § SECTION SIGN
    0x00B6,  # ¶ PILCROW SIGN
    0x2020,  # † DAGGER
    0x2021,  # ‡ DOUBLE DAGGER
    0x25B6,  # ▶ BLACK RIGHT-POINTING TRIANGLE (play button)
    0x25C0,  # ◀ BLACK LEFT-POINTING TRIANGLE
}

def is_emoji_or_symbol(char):
    """Check if a character is an emoji or notable symbol (excluding noise)."""
    cp = ord(char)

    # Skip ignored UI / typography symbols
    if cp in IGNORE_CODEPOINTS:
        return False

    if emoji.is_emoji(char):
        return True
    cat = unicodedata.category(char)
    # So = Symbol, other (includes many pictographs)
    if cat in ("So",):
        return True
    # Check specific Unicode blocks for pictographs/dingbats
    if any([
        0x2600 <= cp <= 0x26FF,   # Miscellaneous Symbols
        0x2700 <= cp <= 0x27BF,   # Dingbats
        0x1F300 <= cp <= 0x1F9FF, # Misc Symbols & Pictographs, Emoticons, etc.
        0x1FA00 <= cp <= 0x1FAFF, # Symbols & Pictographs Extended-A
        0x200D == cp,             # Zero-width joiner (emoji sequences)
    ]):
        return True
    return False

def extract_context(text, pos, window=80):
    """Extract surrounding text context for a character position."""
    start = max(0, pos - window)
    end = min(len(text), pos + window + 1)
    before = text[start:pos].replace("\n", " ")
    after = text[pos+1:end].replace("\n", " ")
    return before, after

# Scan all OCR'd text (already boilerplate-stripped)
findings = []

for filename, text in ocr_results.items():
    for i, char in enumerate(text):
        if is_emoji_or_symbol(char):
            try:
                name = unicodedata.name(char, "UNKNOWN")
            except ValueError:
                name = "UNKNOWN"

            before, after = extract_context(text, i)
            findings.append({
                "file": filename,
                "char": char,
                "codepoint": f"U+{ord(char):04X}",
                "name": name,
                "position": i,
                "context_before": before,
                "context_after": after,
            })

print(f"Total emoji/symbol occurrences found: {len(findings)}")
if len(findings) == 0:
    print("(After stripping DOJ website boilerplate and filtering common UI symbols,")
    print(" no meaningful emoji/symbols remain in this sample.)")
    print("\nConsider:")
    print("  - Increasing SAMPLE_SIZE or processing more datasets")
    print("  - The actual documents may simply not contain emoji codes")

# Save findings
with open(os.path.join(RESULTS_DIR, "emoji_findings.json"), "w") as f:
    json.dump(findings, f, ensure_ascii=False, indent=2)
print(f"Saved to {RESULTS_DIR}/emoji_findings.json")

In [ ]:
# Cell 8: Summary report — frequency table of symbols found
from collections import Counter

if not findings:
    print("No emoji or special symbols were found in the scanned documents.")
else:
    # Count by character
    char_counts = Counter((f["char"], f["codepoint"], f["name"]) for f in findings)
    
    print(f"{'Symbol':<6} {'Codepoint':<12} {'Count':<8} Name")
    print("-" * 60)
    for (char, cp, name), count in char_counts.most_common(50):
        print(f"{char:<6} {cp:<12} {count:<8} {name}")
    
    # Count by file
    file_counts = Counter(f["file"] for f in findings)
    print(f"\n\nFiles with most emoji/symbol occurrences:")
    print(f"{'Count':<8} File")
    print("-" * 50)
    for filename, count in file_counts.most_common(20):
        print(f"{count:<8} {filename}")

In [ ]:
# Cell 9: Show contextual examples for each unique symbol
from collections import defaultdict

if findings:
    by_symbol = defaultdict(list)
    for f in findings:
        by_symbol[f["char"]].append(f)
    
    for char, occurrences in sorted(by_symbol.items(), key=lambda x: -len(x[1])):
        info = occurrences[0]
        print(f"\n{'='*60}")
        print(f"Symbol: {char}  |  {info['codepoint']}  |  {info['name']}")
        print(f"Total occurrences: {len(occurrences)}")
        print(f"{'='*60}")
        
        # Show up to 5 examples with context
        for occ in occurrences[:5]:
            print(f"\n  File: {occ['file']}")
            print(f"  ...{occ['context_before']} [{char}] {occ['context_after']}...")
else:
    print("No findings to display.")

In [ ]:
# Cell 10: Export full results to CSV for further analysis
import csv

if findings:
    csv_path = os.path.join(RESULTS_DIR, "emoji_findings.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "file", "char", "codepoint", "name", "position",
            "context_before", "context_after"
        ])
        writer.writeheader()
        writer.writerows(findings)
    print(f"Exported {len(findings)} findings to {csv_path}")
    
    # Also save the full OCR text for files that had findings
    files_with_findings = set(f["file"] for f in findings)
    relevant_texts = {k: v for k, v in ocr_results.items() if k in files_with_findings}
    with open(os.path.join(RESULTS_DIR, "texts_with_emojis.json"), "w") as f:
        json.dump(relevant_texts, f, ensure_ascii=False, indent=2)
    print(f"Saved full OCR text for {len(relevant_texts)} files with findings")
else:
    print("No findings to export.")

print("\nDone! Review the results above and in the output files.")